In [1]:
#INTERACTIVE VERSION!!!!!!!!!!!!!!!
# version 5 integrates new correlation map, also to add help add width/height filtering and cell grid allignments
#this is v5.py with updated volpy fit changes in 5.3 but without multitrial registration components
#version 4 of test_single_trial_RAM_DISK.py with updated MATLAB .mat saving (sped up)
# TO RUN: conda activate caiman
# # python C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\test_single_trial_RAM_DISK_5.4_simple.py C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B

import argparse
import os
import re
import csv
from datetime import datetime
from pathlib import Path
import traceback

#froot = "G:/INTRSECT_INVIVO/NPCNF139.5R/20251008/FOV5_T4"
froot = r"C:/Users/ICNLab/caiman_data/testdata/testdata/ROOT/NPCNF171.3B/20260203/FOV1_T3"
analysis_mode = "all"  # options: new, old, all


#if re.match(r"^FOV\d+_T\d+$", os.path.basename(froot)):
p = Path(froot)  # normalize to Path
unique_save_string = "-".join(p.parts[-3:])  # last 3 elements
print("Trial folder mode recognized")

# check for prior analysis
#Master CSV path
log_csv_path = Path(froot).parent.parent.parent  / "Analysis" / "MasterAnalysisLOG.csv"

# Ensure the CSV exists with header if needed
if not log_csv_path.exists():
    with open(log_csv_path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Version", "AnalysisDate", "Trial"])
    print("Created new MasterAnalysisLOG.csv with header.")


existing_trials = set()

if log_csv_path.exists():
    with open(log_csv_path, mode="r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_trials.add(row["Trial"])

trial_exists = unique_save_string in existing_trials

if analysis_mode == "new" and trial_exists:
    print("Mode: new -> skipping", p)
elif analysis_mode == "old" and not trial_exists:
    print("Mode: old -> skipping", p)
else:
    # analysis_mode == "all" OR passes mode check
    print("Mode: "+str(analysis_mode)+" -> proceeding:", p)
    folder_paths = froot
    #analyzeFOV([folder_paths], analysis_mode)  # wrap in list for compatibility

# else: #find all folders in MouseID and make list of those folder paths    
#     folder_paths = []
#     base_path = froot
#     for root, dirs, files in os.walk(base_path):
#         for dir_name in dirs:
#             folder_paths.append(os.path.join(root, dir_name))

#     # regex for folders like FOV1_T1, FOV12_T3, etc.
#     pattern = re.compile(r"^FOV\d+_T\d+$")

#     matching_folders = []

#     for folder in folder_paths:
#         for item in os.listdir(folder):
#             item_path = os.path.join(folder, item)
#             if os.path.isdir(item_path) and pattern.match(item):
#                 matching_folders.append(item_path)

#     print(matching_folders)

#     from collections import defaultdict
#     import os
#     import re

#     # group folders by FOV number
#     fov_groups = defaultdict(list)

#     for path in matching_folders:
#         folder_name = os.path.basename(path)
#         match = re.match(r"^FOV(\d+)_T(\d+)$", folder_name)
#         if match:
#             fov_number = match.group(1)  # e.g. "1" from FOV1_T2
#             fov_groups[fov_number].append(path)

#     # sort each FOV group by date and T number
#     for fov in fov_groups:
#         fov_groups[fov].sort(
#             key=lambda p: (
#                 int(os.path.basename(os.path.dirname(p))),  # date: 20250505
#                 int(re.search(r"_T(\d+)$", os.path.basename(p)).group(1))  # trial number
#             )
#         )

#     #Master CSV path
#     log_csv_path = Path(froot).parent / "Analysis" / "MasterAnalysisLOG.csv"
    
#     # Ensure the CSV exists with header if needed
#     if not log_csv_path.exists():
#         with open(log_csv_path, mode='w', newline='') as f:
#             writer = csv.writer(f)
#             writer.writerow(["Version", "AnalysisDate", "Trial"])
#         print("Created new MasterAnalysisLOG.csv with header.")


#     existing_trials = set()

#     if log_csv_path.exists():
#         with open(log_csv_path, mode="r", newline="") as f:
#             reader = csv.DictReader(f)
#             for row in reader:
#                 existing_trials.add(row["Trial"])


#     filtered_fov_groups = {}

#     for fov, paths in sorted(fov_groups.items(), key=lambda x: int(x[0])):

#         kept_paths = []

#         for path in paths:
#             p = path if isinstance(path, Path) else Path(path)

#             unique_save_string = "-".join(p.parts[-3:])
#             trial_exists = unique_save_string in existing_trials  # <-- CSV CHECK

#             if analysis_mode == "new" and trial_exists:
#                 print("Mode: new -> skipping", p)
#                 continue

#             if analysis_mode == "old" and not trial_exists:
#                 print("Mode: old -> skipping", p)
#                 continue

#             # analysis_mode == "all" OR passed checks
#             print("Keeping for analysis:", p)
#             kept_paths.append(p)

#         # Only keep FOVs that still have paths
#         if kept_paths:
#             filtered_fov_groups[fov] = kept_paths

                    
#     fov_groups = filtered_fov_groups



#     for fov, paths in sorted(fov_groups.items(), key=lambda x: int(x[0])):
#         print(f"Analyzing FOV{fov} with {len(paths)} sessions")
#         #analyzeFOV(paths,analysis_mode)



Trial folder mode recognized
Mode: all -> proceeding: C:\Users\ICNLab\caiman_data\testdata\testdata\ROOT\NPCNF171.3B\20260203\FOV1_T3


In [2]:
import matplotlib
matplotlib.use("qtAgg")   # non-interactive, no windows
print(matplotlib.get_backend())

qtAgg


In [3]:


print("Importing packages and Initializing...")
version="V1.2"
#V1.2: 0.8 corr cutoff, 2 minimum ratio of h over w for spikes, cell_idxs incremented by 1, wheel data appended to mat save
print("version:", version)
import matplotlib
matplotlib.use("qtAgg")   # non-interactive, no windows
print(matplotlib.get_backend())
from base64 import b64encode
import cv2
import glob
import h5py
import imageio
from IPython import get_ipython
from IPython.display import HTML, display, clear_output
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from pathlib import Path
from PIL import Image
import re
import csv
from datetime import datetime

#import to cover extras from single_trial.py
import gc
import scipy.io
from scipy import stats
from scipy.signal import butter, lfilter
from scipy.signal import savgol_filter
import sys
import mat73
import pandas as pd


from pathlib import Path

try:
    cv2.setNumThreads(0)
except:
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
        get_ipython().run_line_magic('matplotlib', 'qt')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.utils.utils import download_demo, download_model
from caiman.source_extraction.volpy import utils
from caiman.source_extraction.volpy.volparams import volparams
from caiman.source_extraction.volpy.volpy import VOLPY
from caiman.source_extraction.volpy.mrcnn import visualize, neurons
import caiman.source_extraction.volpy.mrcnn.model as modellib
from caiman.summary_images import local_correlations_movie_offline
from caiman.summary_images import mean_image
from caiman.paths import caiman_datadir
from caiman.summary_images import local_correlations_movie_in_memory
import gc
from caiman.ICNLAB.single_trial_simple_plotting import plotdata

logging.basicConfig(format=
                    "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s]" \
                    "[%(process)d] %(message)s",
                    level=logging.ERROR)




Importing packages and Initializing...
version: V1.2
qtAgg


In [4]:


##BEGIN MAIN ANALYSIS LOOP
#for folder_path in folder_paths:

folder_path = froot


root = Path(folder_path).parts[0] + Path(folder_path).parts[-4] + '\\Analysis\\'

# sub_dir = Path(folder_path).parts[1]

# # Join them with the new folder and filename
# log_path = Path(root) / sub_dir / "Analysis" / "MasterAnalysisLOG.csv"

print(root)
# Output: F:\InVivo\Analysis\MasterAnalysisLOG.csv




C:\ROOT\Analysis\


In [5]:
unique_save_string = "-".join(Path(folder_path).parts[-3:])
print(unique_save_string)

NPCNF171.3B-20260203-FOV1_T3


In [6]:

# find the .tsm file in the folder
tsm_files = [f for f in os.listdir(folder_path) if f.endswith(('.tsm', '.dcimg'))]
if not tsm_files:
    print(f"No recording files found in {folder_path}, skipping.")
#continue if more than one .tsm file found
if len(tsm_files) > 1:
    print(f"Multiple recording files found in {folder_path}, skipping.")

fname = os.path.join(folder_path, tsm_files[0])
print('fname is', fname)
print("Processing file:", fname)

print(folder_path)
fpath = Path(fname)
#Create new unique save name
unique_save_string = "-".join(fpath.parts[-4:-1])
rootpath = str(Path(*fpath.parts[:-4]))+'\\Analysis\\'
print("Unique save string:", unique_save_string)
print("Directory for Analysis Files:", rootpath)
Path(rootpath).mkdir(parents=True, exist_ok=True)
log_csv_path = Path(rootpath) / "MasterAnalysisLOG.csv" #Master CSV path


##
#fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM2\FOV1_T2.tsm'
fr = 640  ################################################################REMOVE LATER
print(fname, fr)


fname is C:/Users/ICNLab/caiman_data/testdata/testdata/ROOT/NPCNF171.3B/20260203/FOV1_T3\FOV1_T3.tsm
Processing file: C:/Users/ICNLab/caiman_data/testdata/testdata/ROOT/NPCNF171.3B/20260203/FOV1_T3\FOV1_T3.tsm
C:/Users/ICNLab/caiman_data/testdata/testdata/ROOT/NPCNF171.3B/20260203/FOV1_T3
Unique save string: NPCNF171.3B-20260203-FOV1_T3
Directory for Analysis Files: C:\Users\ICNLab\caiman_data\testdata\testdata\ROOT\Analysis\
C:/Users/ICNLab/caiman_data/testdata/testdata/ROOT/NPCNF171.3B/20260203/FOV1_T3\FOV1_T3.tsm 640


In [7]:


##
# Cleanup R:/ drive (temp RAM disk)
print("Cleaning up R:/ drive...")
def safe_close_mmap(arr):
    try:
        if hasattr(arr, "base") and hasattr(arr.base, "close"):
            arr.base.close()
    except Exception as e:
        print("close failed:", e)


# 1. Delete any Python references to memmaps pointing to R:/
try:
    safe_close_mmap(Yr)  # or whatever your memmap object is called
except NameError:
    pass

try:
    safe_close_mmap(mmap_file_rig)  # or whatever your memmap object is called
except NameError:
    pass

gc.collect()  # force Python to release the memory mapping

# 2. Delete all files in R:/
for f in Path(r'R:/').glob('*'):
    if f.is_file():
        f.unlink()
print("Cleared all files from R:/")


##
pw_rigid = False  # flag for pw-rigid motion correction
gsig_filt = (3, 3)  # size of filter, in general gSig (see below),
# change this one if algorithm does not work
max_shifts = (5, 5)  # maximum allowed rigid shift
strides = (48, 48)  # start a new patch for pw-rigid motion correction every x pixels
overlaps = (24, 24)  # overlap between paths (size of patch strides+overlaps)
max_deviation_rigid = 3  # maximum deviation allowed for patch with respect to rigid shifts
border_nan = 'copy'
use_cuda = True

opts_dict = {
    'fnames': fname,
    'fr': fr,
    'pw_rigid': pw_rigid,
    'max_shifts': max_shifts,
    'gSig_filt': gsig_filt,
    'strides': strides,
    'overlaps': overlaps,
    'max_deviation_rigid': max_deviation_rigid,
    'border_nan': border_nan,
    'use_cuda': use_cuda
}

opts = volparams(params_dict=opts_dict)

##
print("Loading data...")
m_orig = cm.load(fname)
ds_ratio = 0.2

##
try:
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
except:
    print("Cluster running doing restart")
    dview.terminate()
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
    
##
print("Motion correction...")
mc = MotionCorrect(fname, dview=dview, **opts.get_group('motion'))
mc.motion_correct(save_movie=True, save_dir="R:/")
#about 2.3 minutes for 12800 frames (2m 13-21 s)
print("Done.")

##
print("Loading corrected movie...")
m_rig = cm.load(mc.mmap_file) # 11s
ds_ratio = 0.2
print("Done.")

del m_orig
gc.collect()

####CONVERT VIA STREAMING WITH HOPEFULLY SAME OG BEHAVIOR
p = Path(fname)

ram_path = Path(r'R:/') / (
    f"{p.stem}_rig__d1_{m_rig.shape[1]}"
    f"_d2_{m_rig.shape[2]}"
    f"_d3_1_order_C_frames_{m_rig.shape[0]}.mmap"
)
ram_path = str(ram_path).replace("/", "\\")

# Destination memmap: SAME AS ORIGINAL
dst = np.memmap(
    ram_path,
    dtype='float32',
    mode='w+',
    shape=m_rig.shape,
    order='F'   # critical: this is what caused the layout change originally
)

# Streaming copy (logical copy, not byte copy)
chunk = 16  # frames per chunk; tune for cache / IO

T = m_rig.shape[0]

for t0 in range(0, T, chunk):
    t1 = min(t0 + chunk, T)
    dst[t0:t1] = m_rig[t0:t1]

dst.flush()
mmap_list = [dst]

if hasattr(dst, 'base') and hasattr(dst.base, 'close'):
    dst.base.close()
del dst
gc.collect()

##


Cleaning up R:/ drive...
Cleared all files from R:/
Loading data...
Motion correction...
Saving mmap to:  R:/FOV1_T3_rig__d1_512_d2_512_d3_1_order_F_frames_19200.mmap
Done.
Loading corrected movie...


100%|██████████| 1/1 [00:12<00:00, 12.02s/it]


Done.


0

In [8]:
#check size of array

print(m_rig.shape)


(19200, 512, 512)


In [18]:
#Spectral visualizer

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Ensure the interactive window is active
%matplotlib qt 

# 1. Get dimensions from your existing array
# Assuming m_rig shape is (Time, X, Y)
n_frames, size_x, size_y = m_rig.shape

# Define your sampling frequency (frames per second)
# Change this to your actual recording FPS for accurate Hz
fs = 640

# 2. State Management for interactivity
state = {
    'radius': 10,
    'last_x': size_y // 2, # Start circle in the center
    'last_y': size_x // 2
}

# 3. Setup Visualization
# Note: Matplotlib imshow(A) displays axis 0 as rows (X) and axis 1 as cols (Y)
mean_img = np.mean(m_rig, axis=0)

fig, (ax_img, ax_freq) = plt.subplots(1, 2, figsize=(12, 5))
img_display = ax_img.imshow(mean_img, cmap='viridis', origin='lower')
ax_img.set_title(f"ROI Probe (Shape: {m_rig.shape})")

circle_patch = Circle((state['last_x'], state['last_y']), state['radius'], 
                      color='white', fill=False, lw=1.5)
ax_img.add_patch(circle_patch)

freq_line, = ax_freq.plot([], [], color='firebrick', lw=1)
ax_freq.set_title("Temporal Frequency (FFT of Circular Sum)")
ax_freq.set_xlabel("Frequency (Hz)")
ax_freq.set_ylabel("Amplitude")
ax_freq.grid(True, alpha=0.3)

def update_analysis():
    cx, cy = state['last_x'], state['last_y']
    r = state['radius']
    
    # Update UI elements
    circle_patch.set_center((cx, cy))
    circle_patch.set_radius(r)
    
    # Generate coordinates for masking
    # In imshow, xdata corresponds to columns (axis 2) and ydata to rows (axis 1)
    y_idx, x_idx = np.ogrid[:size_x, :size_y]
    mask = (x_idx - cx)**2 + (y_idx - cy)**2 <= r**2
    
    if np.any(mask):
        # Extract mean temporal signal across the circular ROI
        # m_rig[:, mask] collapses the X,Y dimensions based on the boolean mask
        time_series = m_rig[:, mask].mean(axis=1)
        
        # FFT Calculation (Removing mean to ignore 0Hz/DC offset)
        n = len(time_series)
        yf = np.fft.rfft(time_series - np.mean(time_series))
        xf = np.fft.rfftfreq(n, d=1/fs)
        
        # Update Frequency Plot
        freq_line.set_data(xf, np.abs(yf))
        ax_freq.set_xlim(0, xf.max())
        ax_freq.set_ylim(0, np.max(np.abs(yf)) * 1.1 + 1e-9)
    
    fig.canvas.draw_idle()

def on_move(event):
    if event.inaxes != ax_img: return
    state['last_x'], state['last_y'] = event.xdata, event.ydata
    update_analysis()

def on_scroll(event):
    if event.inaxes != ax_img: return
    # Adjust radius: event.step is usually +1 or -1
    state['radius'] = max(1, state['radius'] + int(event.step * 2))
    update_analysis()

# Connect Interaction Events
fig.canvas.mpl_connect('motion_notify_event', on_move)
fig.canvas.mpl_connect('scroll_event', on_scroll)

plt.tight_layout()
plt.show()


In [22]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.signal import welch  # Added for noise reduction

# %matplotlib qt 

# 1. Dimensions and Config
n_frames, size_x, size_y = m_rig.shape
fs = 640.0 

state = {
    'radius': 20, # Larger radius often helps average out spatial noise too
    'last_x': size_y // 2,
    'last_y': size_x // 2
}

# 2. Setup Plot
mean_img = np.mean(m_rig, axis=0)
fig, (ax_img, ax_freq) = plt.subplots(1, 2, figsize=(13, 5))
ax_img.imshow(mean_img, cmap='magma', origin='lower')
circle_patch = Circle((state['last_x'], state['last_y']), state['radius'], color='cyan', fill=False)
ax_img.add_patch(circle_patch)

freq_line, = ax_freq.plot([], [], color='lime', lw=1.5)
ax_freq.set_yscale('log')
ax_freq.set_facecolor('#111111')
ax_freq.set_xlabel("Frequency (Hz)")
ax_freq.set_ylabel("Power Spectral Density (V^2/Hz)")

def update_analysis():
    cx, cy = state['last_x'], state['last_y']
    r = state['radius']
    circle_patch.set_center((cx, cy))
    circle_patch.set_radius(r)
    
    y_idx, x_idx = np.ogrid[:size_x, :size_y]
    mask = (x_idx - cx)**2 + (y_idx - cy)**2 <= r**2
    
    if np.any(mask):
        # Temporal signal: mean intensity of the circle over time
        time_series = m_rig[:, mask].mean(axis=1)
        
        # --- NOISE REDUCTION: WELCH'S METHOD ---
        # nperseg: length of each segment. Lower = smoother but wider peaks.
        # Try size of n_frames // 4 or // 8 for good smoothing.
        f_axis, psd = welch(time_series - np.mean(time_series), 
                            fs=fs, 
                            nperseg=len(time_series)//4, 
                            scaling='density')
        
        freq_line.set_data(f_axis, psd + 1e-12)
        ax_freq.set_xlim(0.5, fs / 2)
        ax_freq.set_ylim(np.percentile(psd, 5), np.max(psd) * 5)
    
    fig.canvas.draw_idle()

# Event Handlers
def on_move(event):
    if event.inaxes == ax_img:
        state['last_x'], state['last_y'] = event.xdata, event.ydata
        update_analysis()

def on_scroll(event):
    if event.inaxes == ax_img:
        state['radius'] = max(2, state['radius'] + int(event.step * 3))
        update_analysis()

fig.canvas.mpl_connect('motion_notify_event', on_move)
fig.canvas.mpl_connect('scroll_event', on_scroll)
plt.tight_layout()
plt.show()


In [21]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.widgets import SpanSelector
from scipy.signal import welch, butter, filtfilt

# Ensure interactive window
%matplotlib qt 

# 1. Config & Data
n_frames, size_x, size_y = m_rig.shape
fs = 640.0 
state = {
    'radius': 20,
    'last_x': size_y // 2,
    'last_y': size_x // 2,
    'band': [10, 50] # Initial guess for filter band
}

# 2. Setup Main Figure
fig, (ax_img, ax_freq) = plt.subplots(1, 2, figsize=(14, 5))
mean_img = np.mean(m_rig, axis=0)
ax_img.imshow(mean_img, cmap='magma', origin='lower')
ax_img.set_title("1. Scroll: Size | 2. Drag Freq: Band | 3. Click: Filter")

circle_patch = Circle((state['last_x'], state['last_y']), state['radius'], 
                      color='cyan', fill=False, lw=1.5, alpha=0.6)
ax_img.add_patch(circle_patch)

freq_line, = ax_freq.plot([], [], color='lime', lw=1)
ax_freq.set_yscale('log')
ax_freq.set_facecolor('#111111')
ax_freq.set_title("Frequency Selection (Drag to set Band)")

# 3. Bandpass Logic
def apply_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    # Ensure bounds are within Nyquist and valid
    low = max(0.1, low)
    high = min(nyq - 0.1, high)
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, data)

def on_freq_select(xmin, xmax):
    """Called when user drags a span on the frequency plot."""
    state['band'] = [xmin, xmax]
    print(f"Filter Band Set: {xmin:.1f} - {xmax:.1f} Hz")

# SpanSelector for the frequency axis
span = SpanSelector(ax_freq, on_freq_select, 'horizontal', useblit=True,
                    props=dict(alpha=0.5, facecolor='gold'))

def update_analysis(event=None):
    if event and event.inaxes == ax_img:
        state['last_x'], state['last_y'] = event.xdata, event.ydata
    
    cx, cy, r = state['last_x'], state['last_y'], state['radius']
    circle_patch.set_center((cx, cy))
    circle_patch.set_radius(r)
    
    y_idx, x_idx = np.ogrid[:size_x, :size_y]
    mask = (x_idx - cx)**2 + (y_idx - cy)**2 <= r**2
    
    if np.any(mask):
        time_series = m_rig[:, mask].mean(axis=1)
        f, psd = welch(time_series - np.mean(time_series), fs=fs, nperseg=n_frames//4)
        
        freq_line.set_data(f, psd + 1e-12)
        ax_freq.set_xlim(0.5, fs / 2)
        ax_freq.set_ylim(np.percentile(psd, 1), np.max(psd) * 5)
    fig.canvas.draw_idle()

def on_click(event):
    """Extract and plot filtered signal when clicking the image."""
    if event.inaxes != ax_img or event.canvas.manager.toolbar.mode != '':
        return
        
    cx, cy, r = event.xdata, event.ydata, state['radius']
    y_idx, x_idx = np.ogrid[:size_x, :size_y]
    mask = (x_idx - cx)**2 + (y_idx - cy)**2 <= r**2
    
    if np.any(mask):
        raw_signal = m_rig[:, mask].mean(axis=1)
        # Apply the selected bandpass
        filtered = apply_bandpass(raw_signal, state['band'][0], state['band'][1], fs)
        
        # Pop up new window with the result
        plt.figure(figsize=(8, 3))
        t = np.arange(len(raw_signal)) / fs
        plt.plot(t, raw_signal - np.mean(raw_signal), color='gray', alpha=0.5, label='Raw (Zero-mean)')
        plt.plot(t, filtered, color='red', lw=1.5, label=f'Filtered {state["band"][0]:.1f}-{state["band"][1]:.1f}Hz')
        plt.title(f"Signal at ({int(cx)}, {int(cy)})")
        plt.legend()
        plt.xlabel("Time (s)")
        plt.tight_layout()
        plt.show()

# Connect All Events
fig.canvas.mpl_connect('motion_notify_event', update_analysis)
fig.canvas.mpl_connect('scroll_event', lambda e: [setattr(state, 'radius', max(2, state['radius'] + int(e.step*2))), update_analysis()][0])
fig.canvas.mpl_connect('button_press_event', on_click)

plt.tight_layout()
plt.show()


Traceback (most recent call last):
  File "c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\matplotlib\cbook.py", line 298, in process
    func(*args, **kwargs)
  File "C:\Users\ICNLab\AppData\Local\Temp\ipykernel_4580\2896527609.py", line 100, in <lambda>
    fig.canvas.mpl_connect('scroll_event', lambda e: [setattr(state, 'radius', max(2, state['radius'] + int(e.step*2))), update_analysis()][0])
AttributeError: 'dict' object has no attribute 'radius'
Traceback (most recent call last):
  File "c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\matplotlib\cbook.py", line 298, in process
    func(*args, **kwargs)
  File "C:\Users\ICNLab\AppData\Local\Temp\ipykernel_4580\2896527609.py", line 100, in <lambda>
    fig.canvas.mpl_connect('scroll_event', lambda e: [setattr(state, 'radius', max(2, state['radius'] + int(e.step*2))), update_analysis()][0])
AttributeError: 'dict' object has no attribute 'radius'
Traceback (most recent call last):
  File "c:\Users\ICNLab\anaconda3\env

Filter Band Set: 10.6 - 15.3 Hz
Filter Band Set: 11.1 - 15.8 Hz


Traceback (most recent call last):
  File "c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\matplotlib\cbook.py", line 298, in process
    func(*args, **kwargs)
  File "C:\Users\ICNLab\AppData\Local\Temp\ipykernel_4580\2896527609.py", line 100, in <lambda>
    fig.canvas.mpl_connect('scroll_event', lambda e: [setattr(state, 'radius', max(2, state['radius'] + int(e.step*2))), update_analysis()][0])
AttributeError: 'dict' object has no attribute 'radius'
Traceback (most recent call last):
  File "c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\matplotlib\cbook.py", line 298, in process
    func(*args, **kwargs)
  File "C:\Users\ICNLab\AppData\Local\Temp\ipykernel_4580\2896527609.py", line 100, in <lambda>
    fig.canvas.mpl_connect('scroll_event', lambda e: [setattr(state, 'radius', max(2, state['radius'] + int(e.step*2))), update_analysis()][0])
AttributeError: 'dict' object has no attribute 'radius'
Traceback (most recent call last):
  File "c:\Users\ICNLab\anaconda3\env

Filter Band Set: 25.1 - 26.0 Hz


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Config
fs = 640.0
low_freq = 10.0
high_freq = 15.0
n_frames, size_x, size_y = m_rig.shape

# 2. Compute Vectorized FFT along the Time axis (axis=0)
# We subtract the mean per pixel to remove the DC (0Hz) component
print("Computing FFT for all pixels...")
fft_data = np.fft.rfft(m_rig - np.mean(m_rig, axis=0), axis=0)
freqs = np.fft.rfftfreq(n_frames, d=1/fs)

# 3. Find indices for the 6-15 Hz band
idx_band = (freqs >= low_freq) & (freqs <= high_freq)

# 4. Calculate Power (Magnitude Squared) and Sum across the band
# This collapses the 3D FFT result into a 2D Power Map (X, Y)
power_map = np.sum(np.abs(fft_data[idx_band, :, :])**2, axis=0)

# 5. Visualize
plt.figure(figsize=(8, 6))
# Use a logarithmic normalization if the contrast is too high
im = plt.imshow(power_map, cmap='hot', origin='lower') 
plt.colorbar(im, label='Integrated Power (6-15 Hz)')
plt.title(f"Spatial Map of {low_freq}-{high_freq} Hz Variation")
plt.xlabel("X pixels")
plt.ylabel("Y pixels")
plt.show()


Computing FFT for all pixels...


In [24]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq # Scipy FFT is often faster than Numpy

# 1. Ensure float32 to halve memory bandwidth requirements
if m_rig.dtype == np.float64:
    m_rig = m_rig.astype(np.float32)

n_frames, size_x, size_y = m_rig.shape
idx_band = (rfftfreq(n_frames, d=1/640.0) >= 10.0) & (rfftfreq(n_frames, d=1/640.0) <= 15.0)

# 2. Pre-allocate the result map
power_map = np.zeros((size_x, size_y), dtype=np.float32)

# 3. Process in spatial chunks to keep data in CPU Cache
# Even with 190GB RAM, CPU caches (L3) are small; smaller chunks can be faster.
chunk_size = 128 
print("Computing Power Map...")

for i in range(0, size_x, chunk_size):
    # Slice chunk
    chunk = m_rig[:, i:i+chunk_size, :]
    
    # FFT (subtract mean of the chunk to remove DC)
    # workers=-1 uses all available CPU cores
    fft_chunk = rfft(chunk - np.mean(chunk, axis=0), axis=0, workers=-1)
    
    # Magnitude squared and sum band
    power_map[i:i+chunk_size, :] = np.sum(np.abs(fft_chunk[idx_band, :, :])**2, axis=0)
    
    del fft_chunk # Force memory release

plt.imshow(power_map, cmap='hot', origin='lower')
plt.show()


Computing Power Map...


In [28]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq

# 1. Setup
fs = 640.0
low_f, high_f = 12.5, 14
n_frames, nx, ny = m_rig.shape

# Calculate frequency bins once
freqs = rfftfreq(n_frames, d=1/fs)
idx_band = (freqs >= low_f) & (freqs <= high_f)

# 2. Pre-allocate maps
band_power_map = np.zeros((nx, ny), dtype=np.float32)
total_power_map = np.zeros((nx, ny), dtype=np.float32)

# 3. Chunked Processing (Fastest)
chunk_size = 128
print("Computing Normalized Power Map...")

for i in range(0, nx, chunk_size):
    # Slice a spatial chunk
    chunk = m_rig[:, i:i+chunk_size, :].astype(np.float32)
    
    # Subtract mean and compute FFT using all CPU cores
    fft_chunk = rfft(chunk - np.mean(chunk, axis=0), axis=0, workers=-1)
    
    # Calculate Magnitude Squared (|FFT|^2)
    mag_sq = np.abs(fft_chunk)**2
    
    # Store integrated band power (10-15Hz)
    band_power_map[i:i+chunk_size, :] = np.sum(mag_sq[idx_band, :, :], axis=0)
    
    # Store total power (all frequencies except 0Hz)
    total_power_map[i:i+chunk_size, :] = np.sum(mag_sq, axis=0)
    
    del chunk, fft_chunk, mag_sq

# 4. Normalize: Band Power / Total Power
# This shows the FRACTION of variation occurring in the 10-15Hz band
normalized_map = band_power_map / (total_power_map + 1e-12)

# 5. Visualization with Legend (Colorbar)
fig, ax = plt.subplots(figsize=(10, 8))

# Use 'magma' or 'inferno' - they handle dark/dim areas better than 'hot'
im = ax.imshow(normalized_map, cmap='magma', origin='lower')

# Add the Legend (Colorbar)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Fraction of Total Power (10-15 Hz)', rotation=270, labelpad=15)

ax.set_title(f"Normalized Activity Map: {low_f}-{high_f} Hz\n(Highlights specific frequency dominance)")
plt.show()


Computing Normalized Power Map...


In [10]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq

# --- Parameters ---
fs = 640.0
tile_size = 4
min_freq_cutoff = 1.0  # Ignore everything below 1Hz (drifts/noise)
n_frames, height, width = m_rig.shape
new_h, new_w = height // tile_size, width // tile_size

# 1. Spatial Binning (Downsample)
print("Binning tiles...")
m_binned = m_rig.reshape(n_frames, new_h, tile_size, new_w, tile_size).mean(axis=(2, 4))
m_binned = m_binned.astype(np.float32)

# 2. Compute FFT
print("Computing FFT...")
freqs = rfftfreq(n_frames, d=1/fs)
# Subtract mean per tile over time to reduce DC
fft_mag = np.abs(rfft(m_binned - m_binned.mean(axis=0), axis=0))

# 3. Create a mask to ignore low frequencies
# This stops the 0-0.1Hz range from winning
freq_mask = freqs >= min_freq_cutoff
valid_freqs = freqs[freq_mask]
valid_fft = fft_mag[freq_mask, :, :]

# 4. Find Peak Frequency
# We also calculate the max power to mask out "black" (silent) areas
peak_idx = np.argmax(valid_fft, axis=0)
peak_freqs = valid_freqs[peak_idx]
max_power = np.max(valid_fft, axis=0)

# 5. Mask out background (Where power is too low)
# Adjust the percentile (90) if too much or too little is hidden
threshold = np.percentile(max_power, 50) 
peak_freqs[max_power < threshold] = np.nan # Set "quiet" tiles to NaN (transparent/blank)

# 6. Visualization
plt.figure(figsize=(12, 8))
# 'jet' or 'gnuplot' are good for frequency maps
current_cmap = plt.cm.get_cmap('jet').copy()
current_cmap.set_bad(color='black') # Tiles below threshold will be black

im = plt.imshow(peak_freqs, cmap=current_cmap, origin='lower')
cbar = plt.colorbar(im)
cbar.set_label('Peak Frequency (Hz)', rotation=270, labelpad=15)

plt.title(f"Major Peak Frequency Map (Ignoring < {min_freq_cutoff}Hz)")
plt.show()


Binning tiles...
Computing FFT...


C:\Users\ICNLab\AppData\Local\Temp\ipykernel_25896\4038047095.py:43: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  current_cmap = plt.cm.get_cmap('jet').copy()


In [12]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq
from matplotlib.colors import Normalize

# --- Settings ---
fs = 640.0
tile_size = 4
min_freq = 1.0  # Ignore low-freq drift

# 1. Processing (Same as before)
n_frames, h, w = m_rig.shape
new_h, new_w = h // tile_size, w // tile_size
m_binned = m_rig.reshape(n_frames, new_h, tile_size, new_w, tile_size).mean(axis=(2, 4))

freqs = rfftfreq(n_frames, d=1/fs)
fft_mag = np.abs(rfft(m_binned - m_binned.mean(axis=0), axis=0))

# Filter frequencies
mask = freqs >= min_freq
valid_fft = fft_mag[mask, :, :]
peak_freqs = freqs[mask][np.argmax(valid_fft, axis=0)]
max_power = np.max(valid_fft, axis=0)

# 2. Advanced Visualization: Alpha Masking
# Create a base image (e.g., the mean of the first few frames) for context
base_img = m_rig[0:100].mean(axis=0)
base_img_binned = base_img.reshape(new_h, tile_size, new_w, tile_size).mean(axis=(1, 3))

# Normalize power to create an Alpha (Transparency) map
# Power below 'vmin' is invisible; power above 'vmax' is solid color
p_min, p_max = np.percentile(max_power, [10, 95]) # Adjust these to tune the "fade"
alpha_map = np.clip((max_power - p_min) / (p_max - p_min), 0, 1)

# 3. Plotting
fig, ax = plt.subplots(figsize=(12, 10))

# Step A: Plot the grayscale background (anatomy/structure)
ax.imshow(base_img_binned, cmap='gray', origin='lower', alpha=0.8)

# Step B: Create the colored frequency map
# Use 'turbo' (vivid) or 'jet'. 'nipy_spectral' is also great.
cmap = plt.cm.turbo
norm = Normalize(vmin=peak_freqs.min(), vmax=peak_freqs.max())
color_mapped = cmap(norm(peak_freqs))

# Apply the alpha channel to the color map
color_mapped[..., 3] = alpha_map 

# Step C: Overlay the frequency map
im = ax.imshow(color_mapped, origin='lower')

# Manual colorbar since 'im' is now an RGBA array
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Peak Frequency (Hz)')

ax.set_title("Frequency Map with Power-Based Transparency")
plt.show()

# --- After computing peak_freqs and max_power ---

# 1. Logarithmic Normalization for Power
# This 'lifts' the dim areas so they are visible even with lower signal
log_power = np.log10(max_power + 1e-6) 

# 2. Robust Percentile Stretch
# We use the 1st and 99th percentile to ignore outliers and keep fringes visible
p_low, p_high = np.percentile(log_power, [1, 99])
alpha_map = np.clip((log_power - p_low) / (p_high - p_low), 0, 1)

# 3. Create the RGBA Image
# Map peak_freqs to colors, then apply our normalized brightness (Alpha)
norm_freq = Normalize(vmin=peak_freqs.min(), vmax=peak_freqs.max())
color_mapped = plt.cm.turbo(norm_freq(peak_freqs))
color_mapped[..., 3] = alpha_map  # Set brightness via transparency

# 4. Plot over a standard 'Brightened' Mean Image
# We normalize the background image too so the anatomy is clear everywhere
base_img = m_rig.mean(axis=0)
base_img_norm = (base_img - np.min(base_img)) / (np.max(base_img) - np.min(base_img))
# Boost background brightness slightly for context
base_img_norm = np.power(base_img_norm, 0.5) 

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(base_img_norm, cmap='gray', origin='lower') # Structural background
ax.imshow(color_mapped, origin='lower')               # Normalized frequency overlay
plt.show()


In [9]:

print("Computing mean and correlation images...")
img = np.mean(m_rig, axis=0)
img = (img-np.mean(img))/np.std(img)


import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from tqdm import tqdm

# ===============================
# 1. Parameters
# ===============================
HIGHPASS_THRESH = (5)
shape = m_rig.shape
# ===============================
# 2. Load memory-mapped video
# ===============================

video = np.memmap(
    mc.mmap_file[0],
    dtype=np.float32,
    mode="r",
    shape=shape,
    order="C"
).swapaxes(1, 2)

T, H, W = video.shape
print(f"Loaded video: {video.shape}")

# ===============================
# 3. High-pass filter (bandpass-compatible API)
# ===============================
def highpass_filter(data, fs, low, high=None, order=3):
    """
    High-pass filter using the 'low' cutoff.
    The 'high' argument is accepted for API compatibility but ignored.
    """
    nyq = 0.5 * fs
    b, a = butter(order, low / nyq, btype="high")
    return filtfilt(b, a, data, axis=0)

# ===============================
# Parameters
# ===============================
TILE_SIZE = 4
H, W = 512, 512
FRAME_RATE = fr
# (low, high), high ignored
DISPLAY_CLIP = 99

# ===============================
# Coherence metric
# ===============================
def coherence_metric(tile_filt):
    """
    tile_filt: shape (T, Npix)
    Returns mean pixel-to-tile correlation.
    """
    # Tile reference (subthreshold signals sum coherently)
    ref = tile_filt.mean(axis=1)


    ref -= ref.mean()
    ref_std = ref.std() + 1e-9

    # Normalize reference
    ref /= ref_std

    # Normalize pixels
    pix = tile_filt - tile_filt.mean(axis=0)
    pix /= (pix.std(axis=0) + 1e-9)

    # Correlation with reference
    corr = np.mean(ref[:, None] * pix, axis=0)

    # Use mean absolute correlation as coherence
    return np.mean(np.abs(corr))


# ===============================
# Output tile map
# ===============================
n_tiles_y = H // TILE_SIZE
n_tiles_x = W // TILE_SIZE

tile_coherence_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)

# ===============================
# Main loop
# ===============================
with tqdm(total=n_tiles_y * n_tiles_x, desc="Computing coherence") as pbar:
    for ty in range(n_tiles_y):
        for tx in range(n_tiles_x):

            y0 = ty * TILE_SIZE
            y1 = y0 + TILE_SIZE
            x0 = tx * TILE_SIZE
            x1 = x0 + TILE_SIZE

            # Extract tile: (T, 16, 16)
            tile = video[:, y0:y1, x0:x1]
            tile = tile.reshape(T, -1)

            # High-pass filter all pixels independently
            tile_filt = highpass_filter(
                tile, FRAME_RATE, HIGHPASS_THRESH
            )

            # Compute coherence
            tile_coherence_map[ty, tx] = coherence_metric(tile_filt)

            pbar.update(1)

# ===============================
# Expand to image resolution
# ===============================
coherence_image = np.repeat(
    np.repeat(tile_coherence_map, TILE_SIZE, axis=0),
    TILE_SIZE, axis=1
)

if hasattr(video, 'base') and hasattr(video.base, 'close'):
    video.base.close()

del video
gc.collect()


Computing mean and correlation images...
Loaded video: (19200, 512, 512)


Computing coherence: 100%|██████████| 16384/16384 [03:01<00:00, 90.40it/s]


9

In [20]:
#experimental new corr map based on pixel-vs-8-neighbors instead of tile-vs-tile

import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import rfft, irfft, rfftfreq

# 1. Configuration
fs = 640.0
high_pass_cutoff = 5.0
chunk_size = 64  # Smaller chunks often fit better in CPU L3 cache
n_frames, nx, ny = m_rig.shape

# Calculate frequency bins and high-pass mask
freqs = rfftfreq(n_frames, d=1/fs)
hp_mask = freqs >= high_pass_cutoff

def compute_corr_map(mode='1-to-8'):
    print(f"Computing {mode} Correlation Map (High-pass > {high_pass_cutoff}Hz)...")
    corr_map = np.zeros((nx, ny), dtype=np.float32)
    
    # Process in spatial chunks to keep memory bandwidth usage efficient
    for i in range(0, nx, chunk_size):
        i_end = min(i + chunk_size, nx)
        
        # A. Load chunk and High-pass via FFT
        # Subtract mean (DC offset) to avoid spectral leakage
        chunk = m_rig[:, i:i_end, :].astype(np.float32)
        fft_chunk = rfft(chunk - np.mean(chunk, axis=0), axis=0, workers=-1)
        
        # Apply "Brick Wall" High-pass by zeroing low frequencies
        fft_chunk[~hp_mask, :, :] = 0
        
        # Transform back to time domain (this is the filtered signal)
        filtered_chunk = irfft(fft_chunk, n=n_frames, axis=0, workers=-1)
        
        # B. Standardize (Z-score) for fast correlation via Dot Product
        # Correlation(A,B) = dot(z(A), z(B)) / N
        f_mean = np.mean(filtered_chunk, axis=0)
        f_std = np.std(filtered_chunk, axis=0) + 1e-12
        z_chunk = (filtered_chunk - f_mean) / f_std
        
        # C. Correlation Logic
        if mode == '1-to-8':
            # Compare each pixel to its 8 neighbors
            for dx, dy in [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]:
                # Shifted coordinates (with boundary padding)
                # We use np.roll or slicing. Slicing is faster for chunks.
                # For simplicity in chunks, we correlate with a shifted version of the chunk
                shifted = np.roll(z_chunk, shift=(dx, dy), axis=(1, 2))
                corr_map[i:i_end, :] += np.mean(z_chunk * shifted, axis=0)
            corr_map[i:i_end, :] /= 8.0
            
        elif mode == '4x4':
            # Average correlation within 4x4 blocks
            # Reshape and mean to find local "ensemble" similarity
            for r in range(i, i_end):
                for c in range(ny):
                    # Define 4x4 neighborhood bounds
                    r_s, r_e = max(0, r-2), min(nx, r+2)
                    c_s, c_e = max(0, c-2), min(ny, c+2)
                    
                    # Dot product of center pixel with neighborhood
                    center_pixel = z_chunk[:, r-i, c]
                    neighbor_pixels = z_chunk[:, r_s-i:r_e-i, c_s:c_e]
                    
                    # Mean correlation with neighbors
                    corr_map[r, c] = np.mean(np.tensordot(center_pixel, neighbor_pixels, axes=(0,0)))

    return corr_map

# Run and Plot
# Choose '1-to-8' or '4x4'
result_map = compute_corr_map(mode='4x4')

plt.figure(figsize=(10,8))
plt.imshow(result_map, cmap='viridis', origin='lower')
plt.colorbar(label='Avg Pearson Correlation')
plt.title(f"Correlation Map (High-pass > {high_pass_cutoff}Hz)")
plt.show()

coherence_image=result_map

Computing 4x4 Correlation Map (High-pass > 5.0Hz)...


c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


KeyboardInterrupt: 

In [14]:
coherence_image=result_map

print("Computing mean and correlation images...")
img = np.mean(m_rig, axis=0)
img = (img-np.mean(img))/np.std(img)

Computing mean and correlation images...


In [12]:

# ===============================
# Visualization
# ===============================
vmax = np.percentile(coherence_image, DISPLAY_CLIP)

plt.figure(figsize=(6, 6))
plt.imshow(coherence_image, cmap="viridis", vmin=0, vmax=vmax)
plt.title("Grid-based subthreshold coherence ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
plt.colorbar(label="Mean |pixel–tile correlation|")
plt.axis("off")
plt.tight_layout()
plt.show()
#plt.close('all')


NameError: name 'DISPLAY_CLIP' is not defined

In [15]:

img_corr = coherence_image
summary_images = np.stack([img, img, img_corr], axis=0).astype(np.float32)
#cm.movie(summary_images).save(fname[:-5]+'_summary_images.tif')


In [16]:
img = summary_images.transpose([1, 2, 0])


print(fname[:-4]+'_corr.tif')
height, width = img.shape[:2]
print(img.shape)


C:/Users/ICNLab/caiman_data/testdata/testdata/ROOT/NPCNF171.3B/20260203/FOV1_T3\FOV1_T3_corr.tif
(512, 512, 3)


In [11]:
# #optional test to remove bottom x% of pixels from correlation image

# percentile_cutoff = 20  # e.g., remove bottom 20%
# cutoff_value = np.percentile(img_corr, percentile_cutoff)
# img_corr_filtered = np.where(img_corr >= cutoff_value, img_corr, 0)
# summary_images_filtered = np.stack([img, img, img_corr_filtered], axis=0).astype(np.float32)


# #display coherence image and imcorrage filtered
# vmax = np.percentile(img_corr_filtered, DISPLAY_CLIP)
# plt.figure(figsize=(6, 6))
# plt.imshow(img_corr_filtered, cmap="viridis", vmin=0, vmax=vmax
# )
# plt.title("Grid-based subthreshold coherence filtered ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
# plt.colorbar(label="Mean |pixel–tile correlation|")
# plt.axis("off")
# plt.tight_layout()
# plt.show()
# #plt.close('all')   


In [12]:
# #make figure:

# plt.figure(figsize=(10, 5))
# plt.subplot(1, 2, 1)
# plt.imshow(img[0], cmap='gray')
# plt.axis('off')
# #plt.savefig(fname[:-4]+'_mean.tif', format='tif', bbox_inches='tight', pad_inches=0)
# #plt.close('all') # Save the figure and close the plot   

# plt.subplot(1, 2, 2)
# plt.imshow(img[2], cmap='gray')
# plt.axis('off')
# #plt.savefig(fname[:-4]+'_corr.tif', format='tif', bbox_inches='tight', pad_inches=0)
# #plt.close('all') # Save the figure and close the plot   



In [13]:
# print("Computing mean and correlation images...")
# img = np.mean(m_rig, axis=0)
# img = (img-np.mean(img))/np.std(img)

In [14]:
# print(summary_images.shape)
# print(img.shape)
# print(img_corr.shape)



In [17]:
import numpy as np

def get_optimized_blue(img, offset=0.2, gamma=0.5):
    """
    gamma < 1.0 makes the modulation 'stronger' in dark areas (brightens midtones).
    offset ensures a minimum level of visibility.
    """
    r = img[:, :, 0].astype(float)
    b = img[:, :, 2].astype(float)

    # 1. Normalize R to its actual max (instead of 255) to fix the 'dark' issue
    r_max = np.max(r)
    r_norm = r / r_max if r_max > 0 else r

    # 2. Apply Gamma Correction to the mask
    # This lifts the 'weight' of R so it doesn't darken B as aggressively
    r_mask = np.power(r_norm, gamma)

    # 3. Weighted Modulation
    # (1 - offset) ensures the max multiplier is still 1.0
    b_modulated = b * (offset + (1 - offset) * r_mask)

    return np.clip(b_modulated, 0, 255).astype(np.uint8)

# # Try gamma=0.5 to brighten the result significantly
# blue_channel_ready = get_optimized_blue(img, offset=0.1, gamma=0.5)


In [17]:

# --------------------------------------------------------------
# Extract channels like MATLAB
# --------------------------------------------------------------
R = img[:, :, 0]
B = img[:, :, 2] #get_modulated_blue(img)  #img[:, :, 2]

# --------------------------------------------------------------
# MATLAB-style normalization (mat2gray + uint8)
# --------------------------------------------------------------
def normalize_like_matlab(x):
    x = x.astype(np.float64)
    mn = x.min()
    mx = x.max()
    x = (x - mn) / (mx - mn + 1e-12)

    # MATLAB uint8 applies rounding, not floor
    x = np.round(255 * x).astype(np.uint8)
    return x

R_norm = normalize_like_matlab(R)
B_norm = normalize_like_matlab(B)


# --------------------------------------------------------------
# Build MATLAB-equivalent RGB (R,R,B)
# --------------------------------------------------------------
rgb = np.stack([R_norm, R_norm, B_norm], axis=2).astype(np.uint8)
print(rgb.shape)


# --------------------------------------------------------------
# Save as PNG/TIF (MATLAB-compatible pixel data)
# --------------------------------------------------------------
#outname =  rootpath + unique_save_string + ".tif"
#Image.fromarray(rgb).save(outname)

#print("Saved:", outname)
img = rgb.copy()





(512, 512, 3)


In [16]:

# percentile_cutoff = 70 # e.g., remove bottom 20%
# cutoff_value = np.percentile(B_norm, percentile_cutoff)
# img_corr_filtered = np.where(
#     B_norm >= cutoff_value,
#     img_corr,
#     np.zeros_like(img_corr)
# )


# #plot image
# plt.figure(figsize=(6, 6))
# plt.imshow(img_corr_filtered, cmap='gray')
# plt.axis('off')
# plt.tight_layout()
# plt.show()

In [17]:
# B2 = img_corr_filtered
# B2_norm = normalize_like_matlab(B2)

# rgb = np.stack([R_norm, R_norm, B2_norm], axis=2).astype(np.uint8)


In [18]:
# #plot image
# plt.figure(figsize=(6, 6))
# plt.imshow(img[:, :, 2], cmap='gray')
# plt.axis('off')
# plt.tight_layout()
# plt.show()

In [19]:
#plot image
plt.figure(figsize=(6, 6))
plt.imshow(rgb[:, :, 2], cmap='gray')
plt.axis('off')
plt.tight_layout()
plt.show()

In [20]:
# ####SPIKINESS CODE

# import numpy as np
# import matplotlib.pyplot as plt
# from tqdm import tqdm

# # ===============================
# # Parameters
# # ===============================
# video = np.memmap(
#     mc.mmap_file[0],
#     dtype=np.float32,
#     mode="r",
#     shape=shape,
#     order="C"
# ).swapaxes(1, 2)

# T, H, W = video.shape
# print(f"Loaded video: {video.shape}")

# TILE_SIZE = 4
# H, W = 512, 512
# FRAME_RATE = fr   # already defined
# HIGHPASS_THRESH = (30)      # (low, high), high ignored
# DISPLAY_CLIP = 99         # percentile for visualization

# # ===============================
# # Robust spike metric
# # ===============================
# def spikiness_metric(trace, z_thresh=3.0):
#     """
#     Estimate amount of outlying activity in a trace.
#     Uses robust z-score (MAD-based) and measures tail mass.
#     """
#     trace = trace - np.median(trace)
#     mad = np.median(np.abs(trace)) + 1e-9
#     z = trace / mad

#     # Fraction of samples beyond threshold
#     spike_fraction = np.mean(np.abs(z) > z_thresh)

#     # Mean excess magnitude beyond threshold
#     spike_energy = np.mean(np.abs(z[np.abs(z) > z_thresh])) if np.any(np.abs(z) > z_thresh) else 0.0

#     return spike_fraction + 0.1 * spike_energy


# # ===============================
# # Output map (tile-resolution)
# # ===============================
# n_tiles_y = H // TILE_SIZE
# n_tiles_x = W // TILE_SIZE

# tile_spike_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)

# # ===============================
# # Main loop
# # ===============================
# with tqdm(total=n_tiles_y * n_tiles_x, desc="Analyzing tiles") as pbar:
#     for ty in range(n_tiles_y):
#         for tx in range(n_tiles_x):

#             y0 = ty * TILE_SIZE
#             y1 = y0 + TILE_SIZE
#             x0 = tx * TILE_SIZE
#             x1 = x0 + TILE_SIZE

#             # Extract tile: shape (T, 16, 16)
#             tile = video[:, y0:y1, x0:x1]

#             # Sum all pixels
#             tile_trace = tile.reshape(T, -1).sum(axis=1)

#             # High-pass filter (bandpass-compatible)
#             tile_trace_filt = highpass_filter(
#                 tile_trace, FRAME_RATE, HIGHPASS_THRESH
#             )

#             # Compute spikiness
#             tile_spike_map[ty, tx] = spikiness_metric(tile_trace_filt)

#             pbar.update(1)

# # ===============================
# # Expand tile map back to image resolution
# # ===============================
# spike_image = np.repeat(
#     np.repeat(tile_spike_map, TILE_SIZE, axis=0),
#     TILE_SIZE, axis=1
# )

# # # ===============================
# # # Visualization
# # # ===============================
# # vmax = np.percentile(spike_image, DISPLAY_CLIP)

# # plt.figure(figsize=(6, 6))
# # plt.imshow(spike_image, cmap="hot", vmin=0, vmax=vmax)
# # plt.title("Grid-based spike activity ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
# # plt.colorbar(label="Spikiness score")
# # plt.axis("off")
# # plt.tight_layout()
# # plt.show()

# # ===============================
# # Visualization (normalized)
# # ===============================

# # Normalize spike image to [0, 1]
# spike_norm = spike_image.astype(np.float32)
# spike_norm -= spike_norm.min()
# spike_norm /= (spike_norm.max() + 1e-9)  # avoid division by zero

# # Optionally clip at DISPLAY_CLIP percentile for contrast
# vmax = np.percentile(spike_norm, DISPLAY_CLIP / 100 * 1.0)

# plt.figure(figsize=(6, 6))
# plt.imshow(spike_norm, cmap="hot", vmin=0, vmax=vmax)
# plt.title(f"Grid-based spike activity ({TILE_SIZE}×{TILE_SIZE} tiles)")
# plt.colorbar(label="Spikiness score")
# plt.axis("off")
# plt.tight_layout()
# plt.show()

# # ===============================
# # Visualization
# # ===============================
# vmax = np.percentile(spike_image, DISPLAY_CLIP)

# plt.figure(figsize=(6, 6))
# plt.imshow(spike_image, cmap="hot", vmin=0, vmax=vmax)
# plt.title("Grid-based spike activity ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
# plt.colorbar(label="Spikiness score")
# plt.axis("off")
# plt.tight_layout()
# plt.show()

In [21]:
# #spike_image

# p1, p99 = np.percentile(spike_image, (1, 99))
# spikynormed = (spike_image - p1) / (p99 - p1)
# spikynormed = np.clip(spikynormed, 0, 1)

# plt.figure(figsize=(6, 6))
# plt.imshow(spikynormed, cmap="hot", vmin=0, vmax=vmax)
# plt.title("Grid-based spike activity ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
# plt.colorbar(label="Spikiness score")
# plt.axis("off")
# plt.tight_layout()
# plt.show()

In [22]:
# p1, p99 = np.percentile(img, (1, 99))
# spikynormed = (img - p1) / (p99 - p1)
# spikynormed = np.clip(spikynormed, 0, 1)

# plt.figure(figsize=(6, 6))
# plt.imshow(spikynormed, cmap="hot", vmin=0, vmax=vmax)
# plt.title("Grid-based spike activity ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
# plt.colorbar(label="Spikiness score")
# plt.axis("off")
# plt.tight_layout()
# plt.show()

In [23]:
# #plot distribution of pixel values in corr map
# plt.figure(figsize=(6,4))
# plt.hist(spike_image.ravel(), bins=100, color='blue', alpha=0.7)
# plt.title('Distribution of Pixel Values in Correlation Map')
# plt.xlabel('Correlation Value')
# plt.ylabel('Number of Pixels')
# plt.show()


In [24]:
# #plot distribution of pixel values in corr map
# plt.figure(figsize=(6,4))
# plt.hist(B_norm.ravel(), bins=100, color='blue', alpha=0.7)
# plt.title('Distribution of Pixel Values in Correlation Map')
# plt.xlabel('Correlation Value')
# plt.ylabel('Number of Pixels')
# plt.show()


# plt.figure(figsize=(10, 5))
# plt.subplot(1, 2, 1)
# plt.imshow(summary_images[2], cmap='gray')
# plt.axis('off')
# #plt.savefig(fname[:-4]+'_mean.tif', format='tif', bbox_inches='tight', pad_inches=0)
# #plt.close('all') # Save the figure and close the plot   

# plt.subplot(1, 2, 2)
# plt.imshow(B_norm, cmap='gray')
# plt.axis('off')
# #plt.savefig(fname[:-4]+'_corr.tif', format='tif', bbox_inches='tight', pad_inches=0)
# #plt.close('all') # Save the figure and close the plot   
# img = summary_images.transpose([1, 2, 0])

In [21]:


##
print("Running Mask R-CNN inference...")
weights_path="C:/Users/ICNLab/caiman_data/testdata/testdata/mask_rcnn_neuron_0012.h5"
#download_model('mask_rcnn')
#ROIs, r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=False)
ROIs = r['masks'].transpose([2, 0, 1])
Coords = r['rois']
#cm.movie(ROIs).save(fname[:-4]+'newmrcnn_ROIs.hdf5')

fig, axs = plt.subplots(1, 2)
axs[0].imshow(summary_images[1])
axs[1].imshow(ROIs.sum(0))
axs[0].set_title('mean image')
axs[1].set_title('masks')
#plt.savefig(fname[:-6] + 'newmrcnn_ROIs.png', format='png', bbox_inches='tight', pad_inches=0)
#plt.close('all')# Save the figure and close the plot   

#save ROIs as npy array
#np.save(fname[:-4]+'newmrcnn_ROIs.npy', ROIs)
#print("Saved ROIs as npy array:", fname[:-4]+'newmrcnn_ROIs.npy')

###NEW SECTION FOR ROI COORDINATE EXTRACTION
cell_centers = [((y1 + y2) // 2, (x1 + x2) // 2) for (y1, x1, y2, x2) in Coords]
cell_centers = np.array(cell_centers)
print("Cell centers:", cell_centers)    
#display the cell centers on the image
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.imshow(img, cmap='gray') # Display the image
# ax.scatter(cell_centers[:, 1], cell_centers[:, 0], color='red') # Display the cell centers
# ax.set_title('Cell centers')    # Set the title of the plot
#plt.savefig(fname[:-4] + '_cell_centers.png', format='png', bbox_inches='tight', pad_inches=0)
#plt.close('all') # Save the figure and close the plot     

# Save to a file
#save_path = fname[:-4] + '_cell_centers.npy'
#np.save(save_path, cell_centers)

#print(f"Cell centers saved to {save_path}")


Running Mask R-CNN inference...

Configurations:
BACKBONE                       resnet50
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        200
DETECTION_MIN_CONFIDENCE       0
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  512
IMAGE_META_SIZE                14
IMAGE_MIN_DIM                  512
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              crop
IMAGE_SHAPE                    [512 512   3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE                  0.001
LOSS_WEIGHTS                   {'rpn_class_loss': 1.0, 'rpn_bbox_loss': 1.0, 'mrcnn_class_loss': 1.0, 'mrcnn_bbox_loss': 1.0, 'mrcnn_mask_loss': 1.0}
MASK_POOL_SIZE                

     1585758 [deprecation.py:            new_func():554][20392] From c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\tensorflow\python\util\deprecation.py:629: calling map_fn_v2 (from tensorflow.python.ops.map_fn) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Use fn_output_signature instead


Processing 1 images
image                    shape: (512, 512, 3)         min:    0.00000  max:  255.00000  uint8
molded_images            shape: (1, 512, 512, 3)      min:  -91.11000  max:  168.24000  float64
image_metas              shape: (1, 14)               min:    0.00000  max:  512.00000  int32
anchors                  shape: (1, 65472, 4)         min:   -0.04428  max:    1.01297  float32


c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


MADE FIGURE
Cell centers: [[426 114]
 [450 316]
 [161 110]
 [443 146]
 [120 418]
 [400 143]
 [159 189]
 [251 123]
 [157 207]
 [378 271]
 [317  99]
 [221 109]
 [398 215]
 [115  69]
 [108  92]
 [352  74]
 [118 169]
 [428 345]
 [187  40]
 [454 377]
 [403  62]
 [342 172]
 [192  82]
 [ 86 222]
 [122 393]
 [470 183]
 [260  61]
 [299  47]
 [378  53]
 [ 12 351]
 [468  21]
 [330  65]
 [478 363]
 [283  97]
 [300  80]
 [362 196]
 [ 73 248]
 [ 44 275]
 [394 174]
 [ 39 142]
 [ 56  92]
 [382 227]
 [353 487]
 [448 404]
 [ 27 177]
 [335  92]
 [417 440]
 [155 489]
 [155  49]
 [374 168]
 [ 91 187]
 [150 270]
 [416 327]
 [457 442]
 [154 287]
 [247 407]
 [302 338]
 [308 123]
 [ 77 211]
 [417 367]
 [451  95]
 [135 369]
 [ 35 254]
 [197 187]
 [244 433]
 [ 51 343]
 [103 209]
 [394 286]
 [165 175]
 [446 496]]


In [31]:
if ROIs.shape[0] == 0:
    print("No ROIs found.")
else:
    print(f"Found {ROIs.shape[0]} ROIs.")

No ROIs found.


In [27]:

cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False, maxtasksperchild=1)

##
ROIs = ROIs                                   # region of interests
index = list(range(len(ROIs)))                # index of neurons
weights = None                                # if None, use ROIs for initialization; to reuse weights check reuse weights block

template_size = 0.008                         # half size of the window length for spike templates, default is 20 ms
context_size = 35                             # number of pixels surrounding the ROI to censor from the background PCA
visualize_ROI = False                         # whether to visualize the region of interest inside the context region
hp_freq_pb = 1 / 3                            # parameter for high-pass filter to remove photobleaching
clip = 100                                    # maximum number of spikes to form spike template
threshold_method = 'simple'                   # adaptive_threshold or simple
min_spikes= 10                                # minimal spikes to be found
pnorm = 0.5                                   # a variable deciding the amount of spikes chosen for adaptive threshold method
threshold = 4                                 # threshold for finding spikes only used in simple threshold method, Increase the threshold to find less spikes
do_plot = False                               # plot detail of spikes, template for the last iteration
ridge_bg= 0.05                                # ridge regression regularizer strength for background removement, larger value specifies stronger regularization
sub_freq = 20                                 # frequency for subthreshold extraction
weight_update = 'ridge'                       # ridge or NMF for weight update
n_iter = 2                                    # number of iterations alternating between estimating spike times and spatial filters
censor_size = 5                               # size of the censoring region around the ROI
min_width = 0                                 #minumum half peak-height width in ms
max_width = 9                                 #maximum half peak-height width in ms      
w_h_ratio = 2                                 #minumum ratio of height in %dF/F over half peak-height width in ms
                
correl_cutoff = 0.8
snr_thresh_display = 2

opts_dict={'fnames': ram_path,   #'fnames': fname_new,
        'ROIs': ROIs,
        'fr': fr,
        'index': index,
        'weights': weights,
        'min_width': min_width,
        'max_width': max_width,
        'w_h_ratio': w_h_ratio,
        'template_size': template_size,
        'context_size': context_size,
        'visualize_ROI': visualize_ROI,
        'hp_freq_pb': hp_freq_pb,
        'clip': clip,
        'threshold_method': threshold_method,
        'min_spikes':min_spikes,
        'pnorm': pnorm,
        'threshold': threshold,
        'do_plot':do_plot,
        'ridge_bg':ridge_bg,
        'sub_freq': sub_freq,
        'weight_update': weight_update,
        'n_iter': n_iter,
        'censor_size': censor_size}

#opts.change_params(params_dict=opts_dict)
opts = volparams(params_dict=opts_dict)

vpy = VOLPY(n_processes=n_processes, dview=dview, params=opts)

print("Running VOLPY fit...")
vpy.fit(n_processes=n_processes, dview=dview)
#takes a while to run
print("Done.")

# Visualize spatial footprints and traces
#print(np.where(vpy.estimates['locality'])[0])    # neurons that pass locality test
# idx = np.where(vpy.estimates['locality'] > 0)[0]
# utils.view_components(vpy.estimates, img_corr, idx)


##

# Reconstructed movie
# flip_signal = True    
# mv_all = utils.reconstructed_movie(vpy.estimates.copy(), fnames=mc.mmap_file,
#                                         idx=idx, scope=(0,1000), flip_signal=flip_signal)
#mv_all.play(fr=40, magnification=3)

##
vpy.estimates['ROIs'] = ROIs
vpy.estimates['Coords'] = Coords
# save_name = fname[:-4]+'_volpy'
# np.save(save_name, vpy.estimates)

cm.stop_server(dview=dview)
log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)

# print("Saved VOLPY estimates to:", save_name + '.npy')


print(vpy.estimates.keys())
print(len(vpy.estimates['spikes']))
#print(len(vpy.estimates['spikeTimes']))
print(vpy.estimates['snr']) 

#print length of each key's data:
for key in vpy.estimates.keys():
    print(f"{key}: {len(vpy.estimates[key])}")

#print number of neurons with snr > snr_thresh_display
high_snr_neurons = np.sum(vpy.estimates['snr'] > snr_thresh_display)
print(f"Number of neurons with SNR > {snr_thresh_display}: {high_snr_neurons}")


##
vpy = vpy.estimates
#vpy['spikes'] = np.array(vpy['spikes'], dtype=object)




num_frames = np.max(vpy['dFF'].shape)
dur = num_frames/640
vpy['snr_over_thresh'] = []

vpy['raster'] = np.zeros_like(vpy['dFF'])
vpy['firing_rate'] = np.zeros_like(vpy['dFF'])
vpy['unique_trace'] = []
vpy['cell_idxs'] = []

if vpy['spikes'].size > 0:

    for i in range(vpy['dFF'].shape[0]-1):
        vpy['raster'][i, vpy['spikes'][i]] = 1
        vpy['firing_rate'][i] = savgol_filter(np.convolve(vpy['raster'][i]*640,np.ones(32)/32,mode='same'),64,1)

    for i in range(len(vpy['ROIs'])):
        vpy['snr_over_thresh'].append(abs(vpy['snr'][i]) >= snr_thresh_display) #################################################################################################################
    print("SNR LIST", vpy['snr'])
    print("snr_over_thresh", vpy['snr_over_thresh'])
    print("Number of neurons with SNR > 0:", np.sum(vpy['snr_over_thresh']))

    if np.sum(vpy['snr_over_thresh']) > 0:
        to_remove = set()
        dFF = np.array(vpy['dFF']).astype(float)
        R = np.corrcoef(dFF)
        idx0, idx1 = np.where(np.triu(R, 1) > correl_cutoff) #################################################################################################################
        max_vals = np.max(dFF, axis=1)
        smaller = np.where(max_vals[idx0] < max_vals[idx1], idx0, idx1)
        to_remove.update(smaller.tolist())
        vpy['unique_trace'] = [True if x not in to_remove else False for x in range(len(vpy['ROIs']))]

    print(vpy['unique_trace'])
    print("Correl cutoff", correl_cutoff)
    print("There are", np.sum(vpy['unique_trace']), "unique traces after correlation filtering.")
    #print("And there were ", len(to_remove), "traces removed due to high correlation.")

    vpy['cell_idxs'] = []
    for cell in range(len(vpy['ROIs'])):
        if vpy['snr_over_thresh'][cell] and vpy['unique_trace'][cell]:
            vpy['cell_idxs'].append(cell)

    print("Final number of cells after SNR and correlation filtering:", len(vpy['cell_idxs']))
    print(vpy['cell_idxs'])
    print(len(vpy['cell_idxs']))
else:
    print("no spikes")

wheel_mat = os.path.dirname(fname) + '\\Wheel.mat'
if os.path.exists(wheel_mat):
    wheel=mat73.loadmat(wheel_mat)
    print("Loaded wheel data from:", wheel_mat)
else:
    print("No wheel data found at:", wheel_mat)
    wheel = None


Running VOLPY fit...
Starting VOLPY spike detection...


IndexError: list index out of range

In [ ]:
vpy['spikes']

array([array([], dtype=int32),
       array([ 694,  773, 1132, 5261, 5484, 5506, 5814, 7368, 7950, 8013, 8068,
              8532], dtype=int64)                                              ,
       array([3974, 3990, 4247, 4267, 4778, 6735], dtype=int64),
       array([], dtype=int32),
       array([1128, 1163, 1184, 1190, 1290, 1309, 1322, 1328, 1358, 2054, 2107,
              2139, 2162, 2624, 2627, 8969], dtype=int64)                      ,
       array([  360,  5179,  5382,  7828,  7831, 11002], dtype=int64),
       array([  31,  163,  177, 1053, 1071, 1076, 1130, 1137, 1162, 1199, 1204,
              1250, 1256, 1268, 1284, 1307, 2225, 2237, 2247, 2256, 2620, 2983,
              3099, 3104, 3119, 3132, 3138, 3160, 3330, 3337, 3344, 3827, 4049],
             dtype=int64)                                                       ,
       array([], dtype=int32),
       array([1315, 1317, 1356, 1358, 2636, 2638, 2802, 2804, 2975, 5897],
             dtype=int64)                          

In [ ]:
#grab data for plotting from single_trial_simple_plotting.py
mouseID = fpath.parts[-4]
date = fpath.parts[-3]
trialname = fpath.parts[-2]


In [33]:

#make figure
plotdata(vpy, dur, img, ROIs, fname, rootpath, unique_save_string, num_frames, mouseID, date, trialname, wheel)


Wheel behavior data exists but is not in expected format (requires at least 2 columns). Skipping behavior plot.
Wheel behavior shape: (3,)
Saved VOLPY figure to: G:\InVivo\SF132.3B\20250721\FOV1_T3\FOV1_T3_volpy.pdf


In [36]:

print("Saving VOLPY data to MAT file...")
vpy['ROIs'] = ROIs
#vpy['rect'] = r['rois']
vpy['img'] = img
#del vpy['rawROI']
#scipy.io.savemat(fname[:-4] + '_volpy.mat', {'vpy': vpy}, format='5', do_compression=True)

print("Converting data types for fast saving...")

#add 1 to cell_idxs
vpy['cell_idxs'] = [x + 1 for x in vpy['cell_idxs']]

# Keys identified from inspection output that need fixing
keys_to_convert_float = [
    't', 'ts', 't_rec', 't_sub', 'templates', 'snr', 
    'thresh', 'weights', 'locality', 'context_coord', 'F0', 'dFF', 
    'raster', 'firing_rate'
]

keys_to_convert_int = [
    'num_spikes'
]


vpy['wheel'] = wheel #append wheel data to saved mat file

# Load .tbn file to extract downsampled wheel data
tbn_fname = fname[:-4] + ".tbn"
with open(tbn_fname, "rb") as f:
    header = np.fromfile(f, dtype=np.uint8, count=4)   # MATLAB default
    data = np.fromfile(f, dtype=np.float64)
if data.size % 4 != 0:
    raise ValueError(
        f"File has {data.size} float64 values, not divisible by 4"
    )
nrows = data.size // 4
data = data.reshape((nrows, 4), order="F")
# MATLAB: downsample(data(:,4),2)
downsampled_channel_4 = data[:, 3][::2]

vpy['bnc4'] = downsampled_channel_4
print("Extracted downsampled channel 4 from .tbn file and added to vpy['bnc4'].")

# Process float conversions
for key in keys_to_convert_float:
    if key in vpy and vpy[key].dtype == object:
        try:
            # Attempt a direct conversion to float32 (fastest for scientific data)
            vpy[key] = np.array(vpy[key], dtype=np.float32)
            print(f"  Converted '{key}' to float32 array.")
        except ValueError:
            print(f"  Could not convert '{key}' to standard array dtype. Keeping as object array.")

# Process integer conversions
for key in keys_to_convert_int:
    if key in vpy and vpy[key].dtype == object:
        try:
            vpy[key] = np.array(vpy[key], dtype=np.int32)
            print(f"  Converted '{key}' to int32 array.")
        except ValueError:
            print(f"  Could not convert '{key}' to int32 array. Keeping as object array. This is okay.")

# Handle variables that are inherently irregular lists that MUST be object arrays in Python, 
# but we ensure they are clean for saving.

# Handle 'mean_im', 'cell_n', 'polarity' (irregular shapes/strings)
for key in ['mean_im', 'cell_n', 'polarity']:
    if key in vpy and vpy[key].dtype == object:
        vpy[key] = np.array(vpy[key], dtype=object) # Ensure they are formally object arrays

# Handle spikes and low_spikes. The try/except handles the 'bool is not iterable' error.
if vpy['spikes'].dtype == object:
    vpy['spikes'] = np.array([list(x) for x in vpy['spikes']], dtype=object)
    
if vpy['low_spikes'].dtype == object:
    try:
        # This was causing the TypeError because it was actually a boolean array
        vpy['low_spikes'] = np.array([list(x) for x in vpy['low_spikes']], dtype=object)
    except TypeError:
        # If it's a bool array, just make sure it's saved as a clean boolean array
        vpy['low_spikes'] = np.array(vpy['low_spikes'], dtype=bool) 


Saving VOLPY data to MAT file...
Converting data types for fast saving...
Extracted downsampled channel 4 from .tbn file and added to vpy['bnc4'].
  Could not convert 'num_spikes' to int32 array. Keeping as object array. This is okay.


In [35]:

def clean_none(data, name="root"):
    if data is None:
        # Print the name of the key that has the None value
        print(f"Replacing None with [] at: {name}")
        return [] 
    
    elif isinstance(data, dict):
        # Recursively clean each key, passing the key name down for the print statement
        return {k: clean_none(v, name=f"{name} -> {k}") for k, v in data.items()}
    
    elif isinstance(data, list):
        # Recursively clean each list item, passing the index for the print statement
        return [clean_none(v, name=f"{name}[{i}]") for i, v in enumerate(data)]
    
    return data

vpy2 = clean_none(vpy)

print("Data type conversion complete.")

scipy.io.savemat(rootpath + unique_save_string + 'dsafdsafds.mat', {'vpy': vpy2}, format='5', do_compression=True)
#print("Saved VOLPY data to:", fname[:-4] + '_volpy.mat')



Data type conversion complete.


In [6]:
print(wheel.keys())

dict_keys(['data_pos', 'data_time'])


In [9]:
print(fname[:-4])


G:\INTRSECT_INVIVO\NPCNF139.2R\20251125\FOV1_T1\FOV1_T1


In [ ]:
#First draft of .tbn file reading and plotting

# n_samples_param = 512**2          # change this
tbn_fname = fname[:-4] + ".tbn"

# with open(tbn_fname, "rb") as f:
#     header = np.fromfile(f, dtype=np.float64, count=4)
#     nrows = 2 * n_samples_param
#     data = np.fromfile(f, dtype=np.float64, count=nrows * 4)

# data = data.reshape((nrows, 4), order="F")

# downsampled_channel_4 = data[:, 3][::2]
# #channel_4 = data[:, 3]


# nrows = 2 * (512**2)   # invivo{i,8}(3)

# with open(tbn_fname, "rb") as f:
#     # MATLAB fread(fileID,4) → uint8
#     header = np.fromfile(f, dtype=np.uint8, count=4)

#     # MATLAB fread([nrows,4],'float64') → column-major
#     data = np.fromfile(f, dtype=np.float64, count=nrows)
#     data = data.reshape((nrows, 4), order="F")

# downsampled_channel_4 = data[:, 3][::2]
#THIS CODE WORKS FOR ARBITRARY FILE SIZE!!!!!!!!!!!
tbn_fname = fname[:-4] + ".tbn"

with open(tbn_fname, "rb") as f:
    header = np.fromfile(f, dtype=np.uint8, count=4)   # MATLAB default
    data = np.fromfile(f, dtype=np.float64)

if data.size % 4 != 0:
    raise ValueError(
        f"File has {data.size} float64 values, not divisible by 4"
    )

nrows = data.size // 4
data = data.reshape((nrows, 4), order="F")

# MATLAB: downsample(data(:,4),2)
downsampled_channel_4 = data[:, 3][::2]
channel_4 = data[:, 3]

#plot each channel separately
plt.figure(figsize=(10, 8))
for i in range(4):
    plt.subplot(4, 1, i + 1)
    plt.plot(data[:, i], linewidth=1)
    plt.xlabel("Sample index")
    plt.ylabel("Amplitude")
    plt.title(f"Channel {i + 1}")
    plt.grid(True)
    plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(channel_4, linewidth=1)
plt.xlabel("Sample index")
plt.ylabel("Amplitude")
plt.title("Channel 4")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(downsampled_channel_4, linewidth=1)
plt.xlabel("Sample index")
plt.ylabel("Amplitude")
plt.title("Downsampled Channel 4")
plt.grid(True)
plt.tight_layout()
plt.show()





In [16]:

tbn_fname = fname[:-4] + ".tbn"

bytes_per_value = 8          # float64
header_values = 4
n_channels = 4

file_size_bytes = os.path.getsize(tbn_fname)
total_values = file_size_bytes // bytes_per_value

data_values = total_values - header_values
rows = data_values // n_channels

# This is what MATLAB called invivo{i,8}(3)
n_samples_param = rows // 2

print("File size (bytes):", file_size_bytes)
print("Total float64 values:", total_values)
print("Rows detected:", rows)
print("n_samples_param detected:", n_samples_param)

# =========================
# READ FILE
# =========================

with open(tbn_fname, "rb") as f:
    header = np.fromfile(f, dtype=np.float64, count=header_values)
    data = np.fromfile(f, dtype=np.float64, count=rows * n_channels)

data = data.reshape((rows, n_channels), order="F")

# =========================
# PROCESS + PLOT
# =========================

channel_4 = data[:, 3]
downsampled_channel_4 = channel_4[::2]

plt.figure(figsize=(10, 4))
plt.plot(channel_4, label="Channel 4", alpha=0.5)
plt.plot(
    np.arange(0, len(channel_4), 2),
    downsampled_channel_4,
    'o',
    label="Downsampled (×2)",
    markersize=2
)
plt.legend()
plt.title("Sanity check: Channel 4 vs Downsampled")
plt.grid(True)
plt.tight_layout()
plt.show()


File size (bytes): 409604
Total float64 values: 51200
Rows detected: 12799
n_samples_param detected: 6399


In [14]:
print("data dtype:", data.dtype)
print("data shape:", data.shape)

channel_4 = data[:, 3]
downsampled_channel_4 = channel_4[::2]

print("channel_4 dtype:", channel_4.dtype)
print("channel_4 shape:", channel_4.shape)

print("downsampled dtype:", downsampled_channel_4.dtype)
print("downsampled shape:", downsampled_channel_4.shape)


data dtype: float64
data shape: (1024, 4)
channel_4 dtype: float64
channel_4 shape: (1024,)
downsampled dtype: float64
downsampled shape: (512,)


In [37]:

# vpy.estimates['params'] = opts
# save_name = f'volpy_{os.path.split(fnames)[1][:-5]}_{threshold_method}'
# np.save(fnames[:-4] + '_volpy.npy', vpy.estimates)

#del vpy
# % STOP CLUSTER and clean up log files

log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)


# # Cleanup R:/ drive
# print("Cleaning up R:/ drive...")
# def safe_close_mmap(arr):
#     try:
#         if hasattr(arr, "base") and hasattr(arr.base, "close"):
#             arr.base.close()
#     except Exception as e:
#         print("close failed:", e)

# gc.collect()  # force Python to release the memory mapping

# # 2. Delete all files in R:/
# for f in Path(r'R:/').glob('*'):
#     if f.is_file():
#         f.unlink()
# print("Cleared all files from R:/")

#Append new row to MASTERLOG
today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
new_row = [version, today_str, unique_save_string]

with open(log_csv_path, mode='a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(new_row)
print(f"Added new row to MasterAnalysisLOG.csv: {new_row}")


# except Exception as e:
#     print(f"ERROR processing {fname}: {e}")
#     #Append new row to MASTERLOG
#     today_str = datetime.now().strftime("%Y%m%d%H%M%S")  # compact datetime string
#     new_row = [version, today_str, unique_save_string, e]

#     with open(log_csv_path, mode='a', newline='') as f:
#         writer = csv.writer(f)
#         writer.writerow(new_row)
#     print(f"Added new ERROR row to MasterAnalysisLOG.csv: {new_row}")


Added new row to MasterAnalysisLOG.csv: ['V1.2', '20260225152727', 'SF132.3B-20250721-FOV1_T3']
